# HDEM Static — Ablation Study (No Dynamic Weight Update)

This notebook trains HDEM **without** the sliding window dynamic weight update mechanism.
Sub-ensemble weights remain at their initial uniform values (1/K).

Compare results with `HDEM_train.ipynb` to measure the contribution of dynamic weight adaptation.

In [1]:
import os
import sys
sys.path.append(os.path.abspath(".."))
import pandas as pd
import numpy as np
import time
from preprocessing import *
from HDEM_static import Static_Weighted_Ensemble

# set_seed(42)

In [2]:
df = pd.read_csv(r'../output_csv/ANL-Intrepid-2009-1.swf.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68936 entries, 0 to 68935
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   job_id                    68936 non-null  float64
 1   submit_time               68936 non-null  float64
 2   wait_time                 68936 non-null  float64
 3   run_time                  68936 non-null  float64
 4   num_allocated_processors  68936 non-null  float64
 5   avg_cpu_time_used         68936 non-null  float64
 6   used_memory               68936 non-null  float64
 7   requested_processors      68936 non-null  float64
 8   requested_time            68936 non-null  float64
 9   requested_memory          68936 non-null  float64
 10  status                    68936 non-null  float64
 11  user_id                   68936 non-null  float64
 12  group_id                  68936 non-null  float64
 13  executable_id             68936 non-null  float64
 14  queue_

In [3]:
feature_columns = ['requested_processors', 'requested_time', 'submit_time', 'wait_time', 'user_id', 'queue_id']
target_column = 'run_time'

X_train, X_val, X_test, Y_train, Y_val, Y_test, scaler = prepare_data_DL(df, feature_columns, target_column, statuss=-1)

In [4]:
rmse_lst, mae_lst, r2_lst, infer_time_lst = [], [], [], []
for iter in range(5):
    print(f"Iteration {iter + 1} / 100")
    HDEM_static = Static_Weighted_Ensemble(X_train, X_val, X_test, Y_train, Y_val, Y_test, scaler)

    HDEM_static.meta_model_name = 'gradientboosting'
    model_combinations = [['extratrees', 'randomforest', 'xgboost'], ['randomforest', 'mlp', 'gradientboosting'], ['lasso', 'xgboost', 'extratrees']]
    HDEM_static.num_sub = len(model_combinations)
    HDEM_static.init_base_sub(model_combinations)
    sub_ensemble_result = HDEM_static.run_model()
    rmse, mae, r2, infer_time = sub_ensemble_result['RMSE'], sub_ensemble_result['MAE'], sub_ensemble_result['R² Score'], sub_ensemble_result['Inference Time']
    rmse_lst.append(rmse)
    mae_lst.append(mae)
    r2_lst.append(r2)
    infer_time_lst.append(infer_time)

Iteration 1 / 100


Key on Test Set (Static Weights — No Dynamic Update)
Predict on Test Set
MAE:  1377.9283245073086
RMSE:  6278.646005711118
R2:  0.6160087444105129
Inference Time:  2.502521146801124e-06
Iteration 2 / 100
Key on Test Set (Static Weights — No Dynamic Update)
Predict on Test Set
MAE:  1387.015140074042
RMSE:  6302.971891277079
R2:  0.6130275212286482
Inference Time:  2.515181376481968e-06
Iteration 3 / 100
Key on Test Set (Static Weights — No Dynamic Update)
Predict on Test Set
MAE:  1384.7832573434125
RMSE:  6283.695759994644
R2:  0.6153908273453534
Inference Time:  2.593882655020348e-06
Iteration 4 / 100
Key on Test Set (Static Weights — No Dynamic Update)
Predict on Test Set
MAE:  1376.9199861117602
RMSE:  6280.317431101642
R2:  0.6158042741595801
Inference Time:  2.9129865785057957e-06
Iteration 5 / 100
Key on Test Set (Static Weights — No Dynamic Update)
Predict on Test Set
MAE:  1380.89958620077
RMSE:  6267.562969898608
R2:  0.6173631869716957
Inference Time:  3.6039138893715483e-06

In [ ]:
# Save results to CSV
results_df = pd.DataFrame({
    'RMSE': rmse_lst,
    'MAE': mae_lst,
    'R2': r2_lst,
    'Inference Time': infer_time_lst
})
results_df.to_csv('../output_ANL/HDEM_static_results.csv', index=False)

print(f"\n{'='*60}")
print(f"HDEM Static (No Dynamic Weight) — ANL Dataset")
print(f"{'='*60}")
print(f"Mean RMSE: {np.mean(rmse_lst):.4f} ± {np.std(rmse_lst):.4f}")
print(f"Mean MAE:  {np.mean(mae_lst):.4f} ± {np.std(mae_lst):.4f}")
print(f"Mean R²:   {np.mean(r2_lst):.4f} ± {np.std(r2_lst):.4f}")
print(f"Mean Inference Time: {np.mean(infer_time_lst):.6f}s")


HDEM Static (No Dynamic Weight) — ANL Dataset
Mean RMSE: 6282.6388 ± 11.5166
Mean MAE:  1381.5093 ± 3.8805
Mean R²:   0.6155 ± 0.0014
Mean Inference Time: 0.000003s


: 